In [1]:
# =============================================================
# CELL 1 — IMPORTS
# =============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

print("Libraries loaded.")

In [2]:
# =============================================================
# CELL 2 — CONFIGURATION & LOAD SPLIT DATA
# =============================================================
BASE = "https://raw.githubusercontent.com/wip-0/ds207_final_project/main/data/interim/split_groupshuffle/"

# Load Vasileios's pre-split files (target already binarized: 0=NO, 1=readmitted)
X_train = pd.read_csv(BASE + "X_train_mini.csv", na_values="?", low_memory=False)
y_train = pd.read_csv(BASE + "y_train_mini.csv").squeeze()

X_val   = pd.read_csv(BASE + "X_val.csv",   na_values="?", low_memory=False)
y_val   = pd.read_csv(BASE + "y_val.csv").squeeze()

X_test  = pd.read_csv(BASE + "X_test.csv",  na_values="?", low_memory=False)
y_test  = pd.read_csv(BASE + "y_test.csv").squeeze()

print(f"X_train : {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_val   : {X_val.shape}    |  y_val  : {y_val.shape}")
print(f"X_test  : {X_test.shape}   |  y_test : {y_test.shape}")
print(f"\nTarget distribution (train):\n{y_train.value_counts().to_string()}")

In [3]:
# =============================================================
# CELL 3 — DROP UNINFORMATIVE COLUMNS
# =============================================================
# examide & citoglipton: 100% 'No' — zero variance
# weight: 96.9% missing
# payer_code: high missingness, low predictive signal
# encounter_id: identifier only
COLS_TO_DROP = ["encounter_id", "examide", "citoglipton", "weight", "payer_code"]

# Medications with >99% 'No' — drop after association check
MEDS_TO_DROP = [
    "nateglinide", "chlorpropamide", "acetohexamide", "tolbutamide",
    "acarbose", "miglitol", "troglitazone", "tolazamide",
    "glyburide-metformin", "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
]

ALL_DROP = COLS_TO_DROP + MEDS_TO_DROP

def drop_columns(X):
    cols = [c for c in ALL_DROP if c in X.columns]
    return X.drop(columns=cols)

X_train = drop_columns(X_train)
X_val   = drop_columns(X_val)
X_test  = drop_columns(X_test)

print(f"After dropping: X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}")
print(f"Dropped {len(ALL_DROP)} columns: {ALL_DROP}")

In [4]:
# =============================================================
# CELL 4 — FEATURE ENGINEERING
# =============================================================

# ── 4a. Engineered features (per team spec) ──────────────────
def add_engineered_features(X, top_quartile=None):
    X = X.copy()

    # had_emergency: binary flag
    X["had_emergency"] = (X["number_emergency"] > 0).astype(int)

    # high_comorbidity: number_diagnoses >= 75th percentile of TRAIN
    if top_quartile is None:
        top_quartile = X["number_diagnoses"].quantile(0.75)
    X["high_comorbidity"] = (X["number_diagnoses"] >= top_quartile).astype(int)

    # diabetes_dx_count: count of diag cols starting with '250' (BEFORE bucketing)
    for col in ["diag_1", "diag_2", "diag_3"]:
        X[col] = X[col].astype(str).str.strip()
    X["diabetes_dx_count"] = (
        X["diag_1"].str.startswith("250").astype(int) +
        X["diag_2"].str.startswith("250").astype(int) +
        X["diag_3"].str.startswith("250").astype(int)
    )
    return X, top_quartile

# Fit quartile on train only, apply to val/test
X_train, train_q75 = add_engineered_features(X_train)
X_val,   _         = add_engineered_features(X_val,  top_quartile=train_q75)
X_test,  _         = add_engineered_features(X_test, top_quartile=train_q75)

print(f"Engineered features added: had_emergency, high_comorbidity, diabetes_dx_count")
print(f"train_q75 (number_diagnoses): {train_q75}")

# ── 4b. ICD-9 Bucketing ──────────────────────────────────────
def bucket_icd9(code):
    if pd.isna(code) or str(code) in ("nan", ""):
        return "Unknown"
    code = str(code).strip()
    if code.startswith("V"): return "Supplementary"
    if code.startswith("E"): return "External_Cause"
    try:
        num = float(code)
    except ValueError:
        return "Other"
    if 250 <= num < 251:   return "Diabetes"
    if 390 <= num <= 459:  return "Circulatory"
    if 460 <= num <= 519:  return "Respiratory"
    if 520 <= num <= 579:  return "Digestive"
    if 800 <= num <= 999:  return "Injury"
    if 710 <= num <= 739:  return "Musculoskeletal"
    if 580 <= num <= 629:  return "Genitourinary"
    if 140 <= num <= 239:  return "Neoplasms"
    return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    X_train[col] = X_train[col].apply(bucket_icd9)
    X_val[col]   = X_val[col].apply(bucket_icd9)
    X_test[col]  = X_test[col].apply(bucket_icd9)

print("ICD-9 codes bucketed into disease categories.")

# ── 4c. Admission / Discharge / Source Grouping ──────────────
def group_admission_type(val):
    mapping = {1:"Emergency", 2:"Urgent", 3:"Elective",
               4:"Newborn",   5:"NotAvailable", 6:"NotAvailable",
               7:"Trauma",    8:"NotAvailable"}
    return mapping.get(val, "Other")

def group_discharge(val):
    if val == 1:              return "Home"
    if val in [6,8,9,13]:    return "Home_Health"
    if val in [3,4,5]:       return "SNF_Rehab"
    if val in [11,19,20,21]: return "Expired_Hospice"
    if val == 18:             return "Unknown"
    return "Other"

def group_admission_source(val):
    if val == 7:        return "EmergencyRoom"
    if val == 1:        return "PhysicianReferral"
    if val in [2,3]:    return "Transfer"
    return "Other"

def group_medical_specialty(val):
    if pd.isna(val):                          return "Missing"
    val = str(val)
    if "Cardio"          in val:              return "Cardiology"
    if "InternalMedicine" in val:             return "InternalMedicine"
    if "Family"          in val:              return "FamilyPractice"
    if "Surg"            in val:              return "Surgery"
    if "Pulmon"          in val:              return "Pulmonology"
    if "Nephro"          in val:              return "Nephrology"
    if "Endocrin"        in val or "Diabet" in val: return "Endocrinology"
    if "Emergency"       in val:              return "Emergency"
    return "Other"

def group_ids(X):
    X = X.copy()
    X["admission_type_id"]        = X["admission_type_id"].map(group_admission_type)
    X["discharge_disposition_id"] = X["discharge_disposition_id"].map(group_discharge)
    X["admission_source_id"]      = X["admission_source_id"].map(group_admission_source)
    X["medical_specialty"]        = X["medical_specialty"].apply(group_medical_specialty)
    return X

X_train = group_ids(X_train)
X_val   = group_ids(X_val)
X_test  = group_ids(X_test)

print("Admission/discharge/source IDs and medical_specialty grouped.")

# ── 4d. Age ordinal encoding ─────────────────────────────────
age_map = {"[0-10)":0,"[10-20)":1,"[20-30)":2,"[30-40)":3,"[40-50)":4,
           "[50-60)":5,"[60-70)":6,"[70-80)":7,"[80-90)":8,"[90-100)":9}
for X in [X_train, X_val, X_test]:
    X["age"] = X["age"].map(age_map)

print("Age ordinally encoded (0–9).")

# ── 4e. Fill NaN for categorical cols before OHE ─────────────
fill_missing = ["race", "A1Cresult", "max_glu_serum"]
for X in [X_train, X_val, X_test]:
    for col in fill_missing:
        if col in X.columns:
            X[col] = X[col].fillna("Missing")

# ── 4f. Drop gender Unknown/Invalid (only 3 rows) ────────────
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    mask = X["gender"] != "Unknown/Invalid"
    if name == "train":
        X_train, y_train = X[mask].copy(), y[mask].copy()
    elif name == "val":
        X_val, y_val = X[mask].copy(), y[mask].copy()
    else:
        X_test, y_test = X[mask].copy(), y[mask].copy()

print("Gender 'Unknown/Invalid' rows removed.")
print(f"\nFinal shapes — X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

In [5]:
# =============================================================
# CELL 5 — DEFINE COLUMN GROUPS FOR COLUMNTRANSFORMER
# =============================================================

NUMERIC_COLS = [
    "age",                    # ordinal int 0–9
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    # engineered features
    "had_emergency",
    "high_comorbidity",
    "diabetes_dx_count",
]

# Medications kept (NO% < 99%) — OHE with 4 levels (No/Steady/Up/Down)
MEDS_KEPT = [
    "metformin", "repaglinide", "glimepiride", "glipizide",
    "glyburide", "pioglitazone", "rosiglitazone", "insulin",
    "change", "diabetesMed",
]

CATEGORICAL_OHE_COLS = [
    "race", "gender",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "medical_specialty",
    "diag_1", "diag_2", "diag_3",
    "max_glu_serum", "A1Cresult",   # NaN filled with 'Missing'
    *MEDS_KEPT,
]

print(f"Numeric cols  : {len(NUMERIC_COLS)}")
print(f"Categorical cols: {len(CATEGORICAL_OHE_COLS)}")

In [6]:
# =============================================================
# CELL 6 — BUILD SKLEARN PIPELINE
# =============================================================

def build_pipeline():
    """
    Numeric  : median imputation → StandardScaler
    Categorical: most_frequent imputation → OneHotEncoder
    """
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer,  NUMERIC_COLS),
        ("cat", categorical_transformer, CATEGORICAL_OHE_COLS),
    ], remainder="drop")

    return Pipeline([("preprocessor", preprocessor)])

pipeline = build_pipeline()
print("Pipeline built.")

In [7]:
# =============================================================
# CELL 7 — FIT ON TRAIN, TRANSFORM ALL SPLITS
# =============================================================

X_train_proc = pipeline.fit_transform(X_train)   # fit only on train
X_val_proc   = pipeline.transform(X_val)
X_test_proc  = pipeline.transform(X_test)

print(f"X_train_proc : {X_train_proc.shape}")
print(f"X_val_proc   : {X_val_proc.shape}")
print(f"X_test_proc  : {X_test_proc.shape}")

print(f"\ny_train distribution:\n{y_train.value_counts().to_string()}")
print(f"\ny_val distribution:\n{y_val.value_counts().to_string()}")
print(f"\ny_test distribution:\n{y_test.value_counts().to_string()}")
print("\n✅  Preprocessing complete — pipeline ready for modelling.")